In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sqlalchemy import (
    Boolean,
    Column,
    Float,
    Integer,
    MetaData,
    String,
    Table,
    text,
    update,
)

from egomimic.utils.aws.aws_sql import (
    TableRow,
    create_default_engine,
    episode_table_to_df,
)

In [ ]:
engine = create_default_engine()

In [ ]:
df = episode_table_to_df(engine)
df

In [ ]:
import os
from egomimic.scripts.language_process.scale_to_zarr_annotation import (
    build_df_from_tasks,
    get_tasks,
)
from egomimic.utils.scale_utils import get_completed_tasks
project_name = "dense-language"
scale_api_key = "live_5d8ab896bace4cf78c15f2c4f7055306"
tasks = get_completed_tasks(project_name, scale_api_key)
scale_df = build_df_from_tasks(tasks)

In [ ]:
scale_df.columns

In [ ]:
processed_robot_hashes = df[(df["robot_name"] == "eva_bimanual") & (df["task"] == "pick_place") & (df["zarr_processed_path"] != '')]

In [ ]:
processed_robot_hashes

In [ ]:
robot_hashes = df[(df["robot_name"] == "eva_bimanual") & (df["task"] == "pick_place") & (df["zarr_processed_path"] != '')]["episode_hash"].unique()
missing_hashes = [hash for hash in robot_hashes if hash in scale_df["SEQUENCE_ID"].unique()]

In [ ]:
len(missing_hashes)

In [ ]:
len(robot_hashes)

In [ ]:
completed_scale_df = scale_df[scale_df["STATUS"] == "completed"]
hashes = completed_scale_df["SEQUENCE_ID"].unique()

eva_hashes = []
for hash in hashes:
    row = df[df["episode_hash"] == hash]
    if row["robot_name"].iloc[0] == "eva_bimanual":
        eva_hashes.append(hash)
eva_hashes = sorted(eva_hashes)

scale_df[scale_df["SEQUENCE_ID"].isin(eva_hashes)]["STATUS"].value_counts()

In [ ]:
len(eva_hashes)

In [ ]:
duplicates = scale_df[scale_df.duplicated(subset=["SEQUENCE_ID"])]["SEQUENCE_ID"].unique()
eva_hashes_filtered = [hash for hash in eva_hashes if hash not in duplicates]
print(len(eva_hashes_filtered))

In [ ]:
remove_hashes = [
    "2026-03-14-20-06-41-409000",
    "2026-03-14-20-11-52-724000",
    "2026-03-14-20-20-01-584000",
    "2026-03-14-20-38-10-457000",
    "2026-03-14-20-40-25-366000",
    "2026-03-14-20-57-22-308000",
    "2026-03-14-23-13-42-698000",
    "2026-03-15-23-54-15-293000",
    "2026-03-16-02-57-04-244000",
    "2026-03-16-02-59-15-406000",
]
eva_hashes_filtered = [hash for hash in eva_hashes_filtered if hash not in remove_hashes]

In [ ]:
len(eva_hashes_filtered)

In [ ]:
print(str(eva_hashes_filtered))

In [ ]:
scale_df[scale_df["SEQUENCE_ID"] == '2026-03-14-20-40-25-366000']

In [ ]:
scale_df[scale_df["SEQUENCE_ID"] == '2026-03-16-00-06-08-467000']

In [ ]:
# split eva hashes into 4 arrays
eva_hashes_1 = eva_hashes[:len(eva_hashes)//4]
eva_hashes_2 = eva_hashes[len(eva_hashes)//4:len(eva_hashes)//2]
eva_hashes_3 = eva_hashes[len(eva_hashes)//2:3*len(eva_hashes)//4]
eva_hashes_4 = eva_hashes[3*len(eva_hashes)//4:]


In [ ]:
import pandas as pd
def hashes_save_csv(hash, filename):
    df = scale_df[scale_df["SEQUENCE_ID"].isin(hash)]
    print(df)

hashes_save_csv(eva_hashes_1, "eva_hashes_1.csv")